In [ ]:
import sys
sys.path.append("../model")
from gpt import GPT_124
import torch 
import torch.nn as nn
import tiktoken

Mini GPT Configs

In [117]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,   # Vocabulary size
    "context_length": 256, # Shortened context length (orig: 1024)
    "emb_dim": 768,        # Embedding dimension
    "n_heads": 12,         # Number of attention heads
    "n_layers": 12,        # Number of layers
    "drop_rate": 0.1,      # Dropout rate
    "qkv_bias": False      # Query-key-value bias
}

In [118]:
model=GPT_124(GPT_CONFIG_124M)
model.eval()

GPT_124(
  (token_emd): Embedding(50257, 768)
  (pos_emb): Embedding(256, 768)
  (drop_out): Dropout(p=0.1, inplace=False)
  (trf): Sequential(
    (0): Transformer(
      (attn): MultiHeadAttention(
        (W_q): Linear(in_features=768, out_features=768, bias=False)
        (W_k): Linear(in_features=768, out_features=768, bias=False)
        (W_v): Linear(in_features=768, out_features=768, bias=False)
        (dropout): Dropout(p=0.1, inplace=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
      )
      (ff): FeedForward(
        (layer): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNormalization()
      (norm2): LayerNormalization()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): Transformer(
      (attn): MultiHeadAttention(
        (W_q): Linear(in_features=768, out_f

In [119]:
def text_gen(model,ids,max_token,context_size):
    """ 
    ids is input tokens (batch,n_tokens)
    
    """
    for i in range(max_token):
        ids_cl=ids[:,-context_size:] 
        with torch.no_grad():
            out=model(ids_cl)
        last_embd=out[:,-1,:]   
        probs=torch.softmax(last_embd,dim=-1) 
        token=torch.argmax(probs,keepdim=True,dim=-1)
        ids=torch.cat((ids,token),dim=1) 
    return ids    

In [120]:
def text_to_token_ids(text, tokenizer): 
    """ Takes input text converts to tokens and make em ready to pass to gpt"""
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0) # add batch dimension
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0) # remove batch dimension
    return tokenizer.decode(flat.tolist())

    

In [121]:
start_context="Hello dear boy"
tokenizer=tiktoken.get_encoding("gpt2")
ids=text_to_token_ids(start_context, tokenizer)
token_ids = text_gen(
    model=model,
    ids=ids,
    max_token=10,
    context_size=GPT_CONFIG_124M["context_length"]
)
token_ids_to_text(token_ids,tokenizer)

'Hello dear boy paralysis�� Imam recruits foundationalysisfocusedLordound'

In [132]:
inputs = torch.tensor([[16833, 3626, 6100],   # ["every effort moves",
                       [40,    1107, 588]])   #  "I really like"]

targets = torch.tensor([[3626, 6100, 345  ],  # [" effort moves you",
                        [1107,  588, 11311]]) #  " really like chocolate"]

In [123]:
out_embd=model(inputs)
probs=torch.softmax(out_embd,dim=-1)

In [124]:
probs.shape

torch.Size([2, 3, 50257])

In [125]:
tokens=torch.argmax(probs,keepdim=True,dim=-1)
print("Predicted tokens are")
tokens

Predicted tokens are


tensor([[[32785],
         [34076],
         [23230]],

        [[49682],
         [15833],
         [18809]]])

In [126]:
print(f"Target was {tokenizer.decode(targets.squeeze()[0].tolist())} prediction was {tokenizer.decode(tokens.squeeze()[0].tolist())}")
print(f"Target was {tokenizer.decode(targets.squeeze()[1].tolist())} prediction was {tokenizer.decode(tokens.squeeze()[1].tolist())}")

Target was  effort moves you prediction was irlwind furnish Ethiop
Target was  really like chocolate prediction was 893 problematiclav


In [127]:
def loss(logits,target):
    """ 
    out_embd are direct output from gpt -> (batch,n_tokens,n_vocab_size) 
    targets are the target excpected output -> (batch,n_tokens)
    """
    logits_flat = logits.flatten(0, 1)
    target_flat=target.flatten()
    loss=nn.functional.cross_entropy(logits_flat,target_flat)
    return loss


In [133]:
logits=model(inputs)
loss(logits,targets)

tensor(11.0498, grad_fn=<NllLossBackward0>)